# Building a new `cafpybara` analysis (MC only)

Covers setting up a new analysis directory, then loading, plotting, and
cutting a single BNB MC dataframe. For adding a MeVPrtl/HNL (BSM)
sample on top of this, see `step1b_new_analysis_mc_mevprtl.ipynb`.

| Section | File | What you set |
|---|---|---|
| A1 | -- | produce `MC_FILE` via `run_df_maker.py` |
| A2 | `io.py` | `REC_KEY` |
| A3 | `io.py` | `VAR`, `BINS` |
| A4 | `analysis.py` | `signal_categories`, `define_signal` |
| A5 | `analysis.py` | `DEFAULT_CUTS` |
| A6 | `preprocess.py` | new functions (fix/derive variables) |

## 0. Make a new analysis directory

From the `cafpybara/` repo root:

```bash
cp -r analyses/_template analyses/my_analysis
```

* **TODO:** set `CAFPYANA_PATH` in the cell below to your `cafpyana` clone,
  or `export` it before starting Jupyter -- `import cafpybara` raises
  without it.
* **TODO:** change the `_template` import in the cell below to `my_analysis`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the PARENT directory of your cafpybara/ clone -- NOT cafpybara/ itself (skipped if already importable).
sys.path.append("TODO_SET_ME/parent_dir_of_cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

`__init__.py` needs no edits -- copy as-is, it re-exports every `core`
function so each is reachable as `ana.*`. Sanity check:

In [2]:
modules_to_check = ['utils', 'io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']
for mod_name in modules_to_check:
    core_mod = getattr(core, mod_name)
    missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(ana, n)]
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f"core.{mod_name:12s} -> {status}")

## A1. Make `MC_FILE` with `cafpyana`

Prepare your dataframe with `cafpyana`'s `run_df_maker.py`.

This example below uses a `cafpyana` config of a HNL analysis to process MC BNB files without any selections applied:
1. `cd $CAFPYANA_PATH`
2. ```bash
   python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_mcnu_nopreselect_savepfp.py -l <input.list> -o <output_name>
   ```

Already made files from this example can be found here: `/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo`

For more information, see: https://github.com/sungbinoh/cafpyana/blob/lnguyen/hnl_analysis/analysis_village/hnl_nuee_nupi0/README.md

**TODO:** set `MC_FILE`

In [3]:
#MC_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/mcbnb_noselection.df"
MC_FILE = "/path/to/your_mc_file.h5" 

## A2. Load in MC files with `ana.load_mc(...)`

* **TODO:** Set `REC_KEY` in `/my_analysis/io.py`

For the A1 file example, `REC_KEY` is set as `rec`.
See: https://github.com/sungbinoh/cafpyana/blob/lnguyen/hnl_analysis/analysis_village/hnl_nuee_nupi0/configs/hnl_mcnu_nopreselect_savepfp.py#L4


In [4]:
try:
    df, pot, ngen = ana.load_mc(MC_FILE)
    print("MC POT: ", pot)
    print("MC # of events generated:", ngen)
except Exception as e:
    df = None
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## A3. Plot MC with `ana.plot_var`

* **TODO:** set `VAR`, `BINS`


In [7]:
#print out the list of variables in the df
if df is not None:
    for c in df.columns:
        print(c)
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")

In [13]:
#VAR = "your_variable"
#BINS = [0, 1, 2, 3]
VAR = ('pfp', 'shw', 'bestplane_energy', '', '', '') #TEST
BINS = np.linspace(0, 1, 10) #TEST

if df is not None:
    ana.plot_var(df, VAR, bins=BINS)
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")

## A4. Set signal categories in `analysis.py`

* **TODO:** Set `signal_categories` and `signal_dict` in `/my_analysis/analysis.py`

Example:
```python
signal_categories = {
    "CCpi0":        {"value": 0,  "label": r"CC$\nu$$\pi^0$",         "color": "#00CE7C"},
    "NCpi0":        {"value": 1,  "label": r"NC$\nu$$\pi^0$",         "color": "#FF6371"},
    "othernumuCC":  {"value": 2,  "label": r"Other CC $\nu_\mu$",     "color": "#005A8F"},
    "otherNC":      {"value": 3,  "label": r"Other NC $\nu$",         "color": "#DAA8E2"},
    "CCnue":        {"value": 4,  "label": r"CC $\nu_e$",             "color": "#00BBCC"},
    "nonFV":        {"value": 5,  "label": r"Non-FV $\nu$",           "color": "#FF9664"},
    "dirt":         {"value": 6,  "label": r"Dirt $\nu$",             "color": "#8B6969"},
    "cosmic":       {"value": 7,  "label": "Cosmic",                  "color": "#708090"},
    "offbeam":      {"value": 8,  "label": "Offbeam",                 "color": "#D0D2D4"},
}
signal_dict = {k: v["value"] for k, v in signal_categories.items()}
```

* **TODO:** Set `define_signal` conditions in `/my_analysis/analysis.py`

Example:
```python
def define_signal(indf: pd.DataFrame, prefix=None):
    from ...core.utils import ensure_lexsorted

    nudf = ensure_lexsorted(ensure_lexsorted(indf, 0), 1)

    if prefix is None:
        mcdf = nudf
    else:
        mcdf = nudf[prefix]

    # TODO: replace with your own fiducial/active volume bounds
    whereFV = (abs(mcdf.position.x) < 180) & (abs(mcdf.position.y) < 180) & mcdf.position.z.between(10, 480)
    whereAV = (abs(mcdf.position.x) < 200) & (abs(mcdf.position.y) < 200) & mcdf.position.z.between(0, 500)
    whereCCpi0 = ((mcdf.iscc==1)
                & (abs(mcdf.pdg)==14)
                & (mcdf.npi0>0)
                )

    if "signal" in nudf.columns:
        signal = nudf["signal"].to_numpy(copy=True)
    else:
        signal = np.full(len(nudf), -1, dtype=np.int16)

    signal[whereFV & (mcdf.iscc==0) & (mcdf.npi0 > 0)] = signal_dict["NCpi0"]
    signal[whereFV & (mcdf.iscc==1) & (abs(mcdf.pdg)==14) & (mcdf.npi0 == 0)] = signal_dict["othernumuCC"]
    signal[whereFV & (mcdf.iscc==0) & (mcdf.npi0 == 0)] = signal_dict["otherNC"]
    signal[whereFV & (mcdf.iscc==1) & (abs(mcdf.pdg)==12)] = signal_dict["CCnue"]
    signal[whereAV & (signal < 0)] = signal_dict["nonFV"]
    signal[~whereAV] = signal_dict["dirt"]
    signal[np.isnan(mcdf.E)] = signal_dict["cosmic"]

    signal[whereFV & whereCCpi0] = signal_dict["CCpi0"]
    nudf["signal"] = signal
    if ((nudf.signal < 0) | (nudf.signal >= len(signal_dict))).any():
        print("Warning: unidentified signal/background channels present.")
    return nudf
```

In [10]:
print("signal_categories keys:", list(ana.signal_categories.keys()))
print("signal_dict:", ana.signal_dict)

In [ ]:
# What you'd actually write in your own analysis notebook:
if df is not None:
    print(df["signal"].value_counts())
    ana.plot_var(df, VAR, bins=BINS)
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")

## A5. Set cuts in `analysis.py`

* **TODO:** Set `DEFAULT_CUTS` -- a list of `CutSpec`s, applied in order -- in `/my_analysis/analysis.py`

Example: Basic preselection to reject cosmics

```python
DEFAULT_CUTS = [
    CutSpec("clear_cosmic", fn=lambda df: df.slc.is_clear_cosmic == 0,
            label="not clear cosmic"),
    CutSpec("nu_score", variable=("slc", "nu_score"), min=0.5,
            label="nu score > 0.5"),
]
```
* **TODO:** Apply `DEFAULT_CUTS`:
  * Method 1: `load_mc(cuts=...)` -- cuts each chunk while loading; use for
    the final selected sample.
  * Method 2: `select(df, cuts, stage=..., savedict=...)` -- on an
    already-loaded `df`; use to stop at a given cut (`stage=`) or get every
    stage for a cut flow (`savedict=True`).

* **TODO (optional):** build a variant cut list with `modify_cut`/`drop_cuts` for multiple final states

Example: HNL/pi0's `PI0_CUTS_1SHW` vs `PI0_CUTS_2SHW` (`analyses/hnlpi0/analysis.py`)

Example: nueCC's `SIDEBAND_CUTS` (`analyses/nuecc/analysis.py`)

In [ ]:
print("DEFAULT_CUTS:")
for c in ana.DEFAULT_CUTS:
    print(f"  {c.name!r}: {c.label}")

In [ ]:
# Method 1: Apply cuts when loading df
try:
    df_cut, pot_cut, ngen_cut = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)
    ana.plot_var(df_cut, VAR, bins=BINS)
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Method 2: select() on an already-loaded df (stage=/savedict= control how far to apply cuts)
if df is not None:
    #df is the original, uncut dataframe loaded at step A2
    df_dict = ana.select(df, ana.DEFAULT_CUTS, savedict=True)
    for name, df_stage in df_dict.items():
        print(f"  after {name!r}: {len(df_stage)} rows")
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")

### Judging a cut's effect with `plot_var`

`plot_var` has display kwargs useful for comparing before/after a cut --
see `core.plotting.plot_var`'s docstring for the full list.

* `percents=True` -- append each category's %-of-total to its legend
  label; compare pre-/post-cut to see which category a cut removes most.
* `normalize=True` -- rescale each histogram so its integral is 1;
  compare *shape* when a cut also changes the overall rate a lot.
* `counts=True` -- append raw event counts to the legend instead.

See below for the actual call:

In [ ]:
if df is not None:

    #df is the original, uncut dataframe loaded at step A2
    df_dict = ana.select(df, ana.DEFAULT_CUTS, savedict=True)
    df_after = list(df_dict.values())[-1]

    ana.plot_var(df, VAR, bins=BINS, percents=True, title="Before cuts")
    plt.show()

    ana.plot_var(df_after, VAR, bins=BINS, percents=True, title="After cuts")
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")

## A6. Add new functions in `preprocess.py` (Optional)

* **TODO (optional):** Add new functions to `preprocess_mc` in
  `/my_analysis/preprocess.py` to fix existing variables, compute new variables, etc.
  This automatically applies when calling `load_mc`.

Example: Add MC timing calibration (`analyses/hnlpi0/preprocess.py`)

```python
from ...core.preprocess import fix_timing_calibration
from ...core import timing_calibration as tc

def preprocess_mc(df):
    df = _core_preprocess_mc(df)
    df = fix_timing_calibration(df, period=tc.mcbnb_period_calib,
                                 t0_offset=tc.mcbnb_offset_calib)
    return df
```

Description of function`fix_timing_calibration`:
* Input: `slc.barycenterFM.flashTime` + `slc.vertex.z`, plus your
  sample's `period`/`t0_offset` calibration constants
  (`core.timing_calibration`).
* Output: adds `slc.barycenterFM.flashTime_calib` (+ period-folded
  `_calib_mod`) columns.

In [ ]:
# What you'd actually write in your own analysis notebook:
if df is not None:
    VAR = ('slc', 'barycenterFM', 'flashTime_calib_mod', '', '', '') #New variable to plot from the fix_timing_calibration() function
    BINS = np.linspace(0, 19, 20) #TEST

    #df is the original, uncut dataframe loaded at step A2
    df_dict = ana.select(df, ana.DEFAULT_CUTS, savedict=True)
    df_after = list(df_dict.values())[-1]

    ana.plot_var(df_after, VAR, bins=BINS, percents=True, title="After cuts")
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see A2 above")